# Navegar no arquivo ZIM

As células abaixo mostram como listar as entradas (
 / caminhos) dentro do arquivo ZIM e como 
 em subpastas usando o caminho completo. Use `list_children(zim, path)` para ver os nomes imediatos dentro de um caminho e depois passe o caminho completo para acessar uma página específica.

In [1]:
from libzim.reader import Archive
from bs4 import BeautifulSoup
import html as ihtml
import re
import multiprocessing
from tqdm import tqdm
import random
from pprint import pprint
from unidecode import unidecode
import unicodedata
import pandas as pd

In [2]:
random.seed(42)

In [3]:

# Replace 'path/to/your/file.zim' with the actual path to your ZIM file
zim_file_path = 'wikipedia_pt_all_nopic_2025-11.zim'

# Replace "your_file.zim" with the path to your ZIM file
zim = Archive(zim_file_path)

main_entry_path = zim.main_entry.get_item().path
print(f"Main entry is at {main_entry_path}")
# Example 1: Get the main entry's path and size
main_entry = zim.main_entry
if main_entry:
    item = main_entry.get_item()
    print(f"Main entry is at {main_entry.path}, size: {item.size}b.")

Main entry is at Wikipédia:Offline
Main entry is at mainPage, size: 17866b.


In [4]:
entry = zim.get_entry_by_path(main_entry_path)
if entry:
    content_item = entry.get_item()
    # Get the content as bytes, then decode if it's text
    content_bytes = bytes(content_item.content)
    text = content_bytes.decode('UTF-8')

In [5]:
qtde_artigos = zim.all_entry_count

In [6]:
print(f"Total entries in archive: {qtde_artigos}")

Total entries in archive: 2060606


In [7]:
def strip_html_tags_bs4(html_content):
    html_content = ihtml.unescape(html_content)
    soup = BeautifulSoup(html_content, "html.parser")

    paragrafos = soup.find_all("p")

    return ' '.join([p.get_text(separator=" ", strip=True) for p in paragrafos])

def clean_blank_spaces(text):
    clean_bs = re.compile(r'\s+')
    return re.sub(clean_bs, ' ', text).strip()

def process_and_prep(text):
    text = strip_html_tags_bs4(text)
    return clean_blank_spaces(text)

def get_text_by_index(zim, i):
    content_bytes = bytes(zim._get_entry_by_id(i).get_item().content)
    text = content_bytes.decode('UTF-8')
    return text

In [8]:
def limpar_indexes(text):
    return re.sub(r'([\(|\[][\s]?\d{1,8}[\s]?[\)|\]])', ' ', text)

def limpar_caracteres_indesejados(texto):
    return "".join(
        c for c in texto
        if (
            unicodedata.name(c, "").startswith("LATIN")
            or c.isdigit()
            or c.isspace()
            or c in ".,;:!?-/%"
        )
    )

def count_words(text, gram=[2,3,4,5]):
    words = text.split()
    dict_count = {}
    total_ocurrences = 0
    for g in gram:
        for i in range(len(words)-g):
            word_gram = ' '.join(words[i:i+g])
            ocurrences_gram = len(re.findall(re.escape(f'{word_gram}'), text))
            dict_count[word_gram] = ocurrences_gram
            total_ocurrences += ocurrences_gram
        dict_count = dict(
            reversed(sorted(dict_count.items(), key=lambda item: item[1]))
        )
    return dict_count, total_ocurrences

def remover_longos_numeros(text):
    return re.sub(r'\d{5,}', ' ', text)

def arrumar_pontuacoes(text):
    return re.sub(r'\s+([.,;:!?%])', r'\1', text)

def arrumar_espacos_da_divisao(text):
    return re.sub(r'([\d|\w])*\s([\/])\s([\d|\w]*)', r'\1\2\3', text)

def arrumar_espacos(text):
    return re.sub(r'\s+', ' ', text)


In [9]:
text_amostra = get_text_by_index(zim, 556489)
pprint(text_amostra)

('<!DOCTYPE html>\n'
 '<html class="client-nojs vector-feature-language-in-header-enabled '
 'vector-feature-language-in-main-page-header-disabled '
 'vector-feature-page-tools-pinned-disabled '
 'vector-feature-toc-pinned-clientpref-0 vector-toc-not-available '
 'vector-feature-main-menu-pinned-disabled '
 'vector-feature-limited-width-clientpref-1 '
 'vector-feature-limited-width-content-enabled '
 'vector-feature-custom-font-size-clientpref-1 '
 'vector-feature-appearance-pinned-clientpref-0 '
 'vector-feature-night-mode-enabled skin-theme-clientpref-os '
 'vector-sticky-header-enabled" lang="pt" dir="ltr"><head>\n'
 '    <meta charset="UTF-8">\n'
 '    <title>Daniel Agrobom</title>\n'
 '    <meta name="viewport" content="width=device-width, initial-scale=1.0">\n'
 '    <link rel="icon" type="image/png" href="./_res_/favicon.png">\n'
 '    <link rel="canonical" '
 'href="https://pt.wikipedia.org/wiki/Daniel_Agrobom"> <link '
 'href="./_mw_/ext.cite.styles.css" rel="stylesheet" type=

In [10]:
def clean_text(text):
    text_amostra_cleaned = strip_html_tags_bs4(text)
    text_amostra_cleaned = limpar_indexes(text_amostra_cleaned)
    text_amostra_cleaned = clean_blank_spaces(text_amostra_cleaned)
    text_amostra_cleaned = limpar_caracteres_indesejados(text_amostra_cleaned)
    text_amostra_cleaned = remover_longos_numeros(text_amostra_cleaned)
    text_amostra_cleaned = arrumar_pontuacoes(text_amostra_cleaned)
    text_amostra_cleaned = arrumar_espacos(text_amostra_cleaned)
    text_amostra_cleaned =  arrumar_espacos_da_divisao(text_amostra_cleaned)
    return text_amostra_cleaned.strip()

In [11]:
def worker(_):
    try:
        index = random.randrange(qtde_artigos)
        text = get_text_by_index(zim, index)
        text_cleaned = clean_text(text)
        if len(text_cleaned) > 200:
            return text_cleaned, index 
    except UnicodeDecodeError:
        pass
    return None

In [ ]:

list_texts = []
target = 1_000_000
workers = 8
print(f'Rodando em {workers=}')

with multiprocessing.Pool(workers) as pool:
    with tqdm(total=target, desc="Coletando textos") as pbar:
        for result in pool.imap_unordered(worker, range(target * 2)):
            if result is not None:
                list_texts.append({'index': result[1], 'texto': result[0]})
                pbar.update(1)

            if len(list_texts) >= target:
                pool.terminate()
                break

Rodando em workers=8


Coletando textos:  27%|██▋       | 272950/1000000 [13:32<37:11, 325.78it/s]  /tmp/ipykernel_7091/847739366.py:3: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  soup = BeautifulSoup(html_content, "html.parser")
Coletando textos:  30%|███       | 300468/1000000 [14:51<37:12, 313.32it/s]/tmp/ipykernel_7091/847739366.py:3: X

In [ ]:
file_export = 'data/base_wiki_pt_cleaned_2.pq'

In [25]:
list_texts[:2]

[{'index': 1031610,
  'texto': 'O Opirus é um automóvel sedan de porte grande da Kia. Como a primeira entrada da Kia para o grande carro mercado, o Opirus/Amanti tinha sido comercializado em um único nível de acabamento e apenas como um sedan. Ele compartilhou alguns componentes com o seu primo incorporado agora extinta, a Hyundai Grandeur XG, incluindo a sua 3.5 L V6. Para 2007, o Kia Opirus recebeu vários upgrades, incluindo a suspensão e revisão de estilo, e a adição do mesmo motor que o atual Hyundai Azera, desta vez sendo um 3,8 L V6. Nos EUA, o Opirus foi reconhecida como a mais atraente premium Midsize Car pela JD Power and Associates 2005 Desempenho Automotivo, Execução e Estudo de Layout. O Opirus 2007 superaram vários carros de luxo no Instituto de Seguros para a Segurança Rodoviária IIHS testes de colisão de impacto lateral, para ganhar a mais alta classificação do bem. A partir de 17 de dezembro de 2010, o site da Kia já não listou mais o Opirus como um modelo de produção. 

In [ ]:
df = pd.DataFrame(list_texts)

In [ ]:
df.head()

,index,texto
0,1031610,O Opirus é um automóvel sedan de porte grande ...
1,1895765,Costa Macedo Giraldes Barba de Noronha e Brito...
2,649783,Eleutherodactylus cajamarcensis é uma espécie ...
3,1826843,Tricromatismo ou visão tricromática é a capaci...
4,1351999,Ode do grego antigo ōidē é um poema de estilo ...


In [27]:
len(df)

1000000

In [ ]:
df.to_parquet(file_export, index=False)

In [ ]:
df = pd.read_parquet(file_export, engine='fastparquet')

In [33]:
pprint([t for t in df.sample(10)['texto'].values])

['Muammar Muhammad Abu Minyar al-Gaddafi a Abu Hadi, c. 1942 Sirte, 20 de '
 'outubro de 2011, vulgarmente conhecido como Coronel Gaddafi, foi um '
 'revolucionário líbio, político e teórico político. Governou a Líbia como '
 'Presidente Revolucionário da República Árabe Líbia de 1969 a 1977 e depois '
 'como Líder Fraternal da Grande Jamahiriya Árabe Popular Socialista da Líbia '
 'de 1977 a 2011. Inicialmente estava ideologicamente empenhado no '
 'nacionalismo árabe e no socialismo árabe, mas mais tarde governou conforme a '
 'sua própria Terceira Teoria Internacional. Nascido perto de Sirte, Líbia '
 'Italiana, numa família beduína pobre, Gaddafi tornou-se um nacionalista '
 'árabe enquanto frequentava a escola em Sabha, matriculando-se mais tarde na '
 'Academia Militar Real, Benghazi. No seio das forças armadas, fundou um grupo '
 'revolucionário que depôs a monarquia Senussi, apoiada pelo Ocidente, de '
 'Idris I, num golpe de Estado em 1969. Tendo tomado o poder, Gaddafi '
 'co

In [ ]:
\\